# 🔬 Hyperparameter Tuning MobileNetV2 — Stratified K-Fold CV

**Metode:** Control Variable — ubah 1 variabel per tahap, kunci yang terbaik, lanjut ke berikutnya.

| Tahap | Parameter | Variasi yang Diuji |
|---|---|---|
| 0 | Baseline | lr=1e-3, bs=32, dropout=0.2, epochs=30, aug=baseline |
| 1 | Learning Rate | 1e-3, 1e-4, 1e-5 |
| 2 | Batch Size | 16, 32, 64 |
| 3 | Dropout Rate | 0.2, 0.3, 0.5 |
| 4 | Max Epochs | 15, 30, 50 |
| 5 | Augmentation | minimal, baseline, enhanced |

**Total: 80 training runs** (5 fold × 16 konfigurasi)  
**Estimasi waktu (GPU): ~2-3 jam**  
**Resume support:** Ya — jika terputus, jalankan ulang, tahap yang sudah selesai akan di-skip.

---

## Cara Pakai
1. Upload folder `gambar_balanced/` sebagai Kaggle Dataset
2. Nyalakan **GPU** (Settings → Accelerator → GPU T4 x2 atau P100)
3. Add dataset yang sudah di-upload
4. **Sesuaikan `DATASET_DIR`** di cell berikutnya
5. Run All Cells

## 1. Setup — Imports & Konfigurasi

In [4]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score, recall_score
)
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import gc
import time
import json

# ============================================================
# KONFIGURASI — SESUAIKAN SEBELUM MENJALANKAN
# ============================================================

# ⬇️ SESUAIKAN PATH INI DENGAN DATASET ANDA DI KAGGLE ⬇️
DATASET_DIR = "/kaggle/input/gambar-balanced/gambar_balanced"

# Path output (tidak perlu diubah)
OUTPUT_DIR = "/kaggle/working/hasil_tuning_vision"

# Parameter tetap
IMG_SIZE = (224, 224)
K_FOLDS = 5
SEED = 42

# Ekstensi gambar yang valid
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# Reproducibility
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU check
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU ditemukan: {len(gpus)} device(s)')
else:
    print('⚠ Tidak ada GPU — akan menggunakan CPU (jauh lebih lambat)')

print(f'TensorFlow version: {tf.__version__}')
print(f'Dataset: {DATASET_DIR}')
print(f'Output : {OUTPUT_DIR}')

✅ GPU ditemukan: 2 device(s)
TensorFlow version: 2.20.0
Dataset: /kaggle/input/gambar-balanced/gambar_balanced
Output : /kaggle/working/hasil_tuning_vision


## 2. Load Dataset ke Memory

In [5]:
def load_dataset(dataset_dir, img_size=(224, 224)):
    """
    Memuat seluruh gambar dari folder dataset ke numpy array.
    800 gambar x 224x224x3 ≈ 460 MB — aman untuk Kaggle (16GB RAM).
    """
    images = []
    labels = []
    class_names = sorted([
        d for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ])
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}

    print(f'\n📂 Memuat dataset dari: {dataset_dir}')

    for class_name in class_names:
        class_dir = os.path.join(dataset_dir, class_name)
        count = 0
        for img_name in sorted(os.listdir(class_dir)):
            if os.path.splitext(img_name)[1].lower() not in VALID_EXTENSIONS:
                continue
            img_path = os.path.join(class_dir, img_name)
            try:
                img = tf.keras.utils.load_img(img_path, target_size=img_size)
                img_array = tf.keras.utils.img_to_array(img)
                images.append(img_array)
                labels.append(class_to_idx[class_name])
                count += 1
            except Exception as e:
                print(f'  ⚠ Skip {img_path}: {e}')
        print(f'  {class_name}: {count} gambar')

    X = np.array(images, dtype=np.float32)
    y = np.array(labels)
    print(f'\n📊 Total: {len(X)} gambar, {len(class_names)} kelas')
    print(f'   Shape X: {X.shape}')
    print(f'   Distribusi: {dict(zip(class_names, np.bincount(y)))}')

    return X, y, class_names


# ============================================================
# AUTO-DETECT PATH DATASET DI KAGGLE
# ============================================================
print("📂 Isi /kaggle/input/:")
for item in os.listdir("/kaggle/input/"):
    full = os.path.join("/kaggle/input/", item)
    print(f"  /{item}/ → {os.listdir(full)[:5]}...")

# Cari path yang berisi folder-folder komoditas
def find_dataset_path(base="/kaggle/input"):
    target_classes = {'bawang_merah', 'bawang_putih', 'beras', 'cabai_merah',
                      'cabai_rawit', 'daging_ayam', 'minyak_goreng', 'telur_ayam'}
    for root, dirs, files in os.walk(base):
        if target_classes.issubset(set(dirs)):
            return root
    return None

detected_path = find_dataset_path()
if detected_path:
    DATASET_DIR = detected_path
    print(f"\n✅ Dataset ditemukan di: {DATASET_DIR}")
else:
    print(f"\n❌ Dataset tidak ditemukan! Cek struktur folder di sidebar Kaggle.")

# Load dataset
X, y, class_names = load_dataset(DATASET_DIR, IMG_SIZE)

📂 Isi /kaggle/input/:
  /datasets/ → ['aksh889']...

✅ Dataset ditemukan di: /kaggle/input/datasets/aksh889/gambar-balanced/gambar_balanced

📂 Memuat dataset dari: /kaggle/input/datasets/aksh889/gambar-balanced/gambar_balanced
  bawang_merah: 100 gambar
  bawang_putih: 100 gambar
  beras: 100 gambar
  cabai_merah: 100 gambar
  cabai_rawit: 100 gambar
  daging_ayam: 100 gambar
  minyak_goreng: 100 gambar
  telur_ayam: 100 gambar

📊 Total: 800 gambar, 8 kelas
   Shape X: (800, 224, 224, 3)
   Distribusi: {'bawang_merah': np.int64(100), 'bawang_putih': np.int64(100), 'beras': np.int64(100), 'cabai_merah': np.int64(100), 'cabai_rawit': np.int64(100), 'daging_ayam': np.int64(100), 'minyak_goreng': np.int64(100), 'telur_ayam': np.int64(100)}


## 3. Definisi Fungsi — Model, K-Fold, & Utilitas

In [6]:
def build_model(num_classes, learning_rate=1e-3, dropout_rate=0.2,
                augmentation_type='baseline'):
    """
    Membangun arsitektur MobileNetV2 + Transfer Learning.
    Model dibangun dari scratch untuk setiap fold (menghindari weight leakage).
    """

    # Layer Data Augmentation
    if augmentation_type == 'minimal':
        aug_layers = [
            tf.keras.layers.RandomFlip('horizontal'),
        ]
    elif augmentation_type == 'enhanced':
        aug_layers = [
            tf.keras.layers.RandomFlip('horizontal'),
            tf.keras.layers.RandomRotation(0.2),
            tf.keras.layers.RandomZoom(0.2),
            tf.keras.layers.RandomBrightness(0.2),
            tf.keras.layers.RandomContrast(0.2),
        ]
    else:  # baseline
        aug_layers = [
            tf.keras.layers.RandomFlip('horizontal'),
            tf.keras.layers.RandomRotation(0.2),
            tf.keras.layers.RandomZoom(0.2),
        ]

    data_augmentation = tf.keras.Sequential(aug_layers, name='data_augmentation')

    # Base Model (Frozen)
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    # Arsitektur
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = data_augmentation(inputs)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model


def run_kfold(X, y, class_names, params, config_name, stage_dir):
    """
    Menjalankan Stratified K-Fold Cross-Validation untuk satu set parameter.
    """

    skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
    fold_results = []
    all_y_true = []
    all_y_pred = []

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        fold_num = fold_idx + 1
        print(f'    Fold {fold_num}/{K_FOLDS}...', end=' ', flush=True)
        start_time = time.time()

        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Build model baru dari scratch (hindari weight leakage)
        tf.keras.backend.clear_session()
        tf.random.set_seed(SEED + fold_idx)
        np.random.seed(SEED + fold_idx)

        model = build_model(
            num_classes=len(class_names),
            learning_rate=params['learning_rate'],
            dropout_rate=params['dropout_rate'],
            augmentation_type=params.get('augmentation', 'baseline')
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=0
        )

        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
        prec = precision_score(y_val, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_val, y_pred, average='macro', zero_division=0)
        epochs_run = len(history.history['loss'])
        best_val_loss = min(history.history['val_loss'])
        best_val_acc = max(history.history['val_accuracy'])

        elapsed = time.time() - start_time
        print(f'Acc={acc:.4f}  F1={f1:.4f}  Epochs={epochs_run}  ({elapsed:.1f}s)')

        fold_results.append({
            'config': config_name,
            'fold': fold_num,
            'accuracy': round(acc, 4),
            'f1_macro': round(f1, 4),
            'precision_macro': round(prec, 4),
            'recall_macro': round(rec, 4),
            'epochs_run': epochs_run,
            'best_val_loss': round(best_val_loss, 4),
            'best_val_acc': round(best_val_acc, 4),
            'time_seconds': round(elapsed, 1)
        })

        all_y_true.extend(y_val.tolist())
        all_y_pred.extend(y_pred.tolist())

        del model, history
        gc.collect()

    # Confusion matrix gabungan
    cm = confusion_matrix(all_y_true, all_y_pred)
    _save_confusion_matrix(cm, class_names, config_name, stage_dir)

    return fold_results


def _save_confusion_matrix(cm, class_names, config_name, output_dir):
    """Simpan heatmap confusion matrix ke PNG."""
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names, ax=ax
    )
    ax.set_xlabel('Prediksi', fontsize=12)
    ax.set_ylabel('Aktual', fontsize=12)
    ax.set_title(f'Confusion Matrix — {config_name}', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()

    safe_name = config_name.replace(' ', '_').replace('=', '_')
    fig.savefig(os.path.join(output_dir, f'confusion_matrix_{safe_name}.png'), dpi=150)
    plt.close(fig)


def _save_comparison_chart(recap_df, stage_name, output_dir):
    """Buat bar chart perbandingan accuracy & F1-score."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    configs = recap_df['config'].tolist()
    x = np.arange(len(configs))

    best_acc_pos = recap_df['mean_accuracy'].values.argmax()
    best_f1_pos = recap_df['mean_f1_macro'].values.argmax()

    colors_acc = ['#2E7D32' if i == best_acc_pos else '#A5D6A7' for i in range(len(configs))]
    colors_f1 = ['#1565C0' if i == best_f1_pos else '#90CAF9' for i in range(len(configs))]

    # Accuracy
    axes[0].bar(x, recap_df['mean_accuracy'], yerr=recap_df['std_accuracy'],
                capsize=5, color=colors_acc, edgecolor='black', linewidth=0.5)
    axes[0].set_ylabel('Accuracy', fontsize=11)
    axes[0].set_title('Accuracy (Mean ± Std)', fontsize=12, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(configs, rotation=45, ha='right', fontsize=10)
    y_min_acc = max(0, recap_df['mean_accuracy'].min() - 0.15)
    axes[0].set_ylim(y_min_acc, 1.05)
    for i, (v, s) in enumerate(zip(recap_df['mean_accuracy'], recap_df['std_accuracy'])):
        label = f'{v:.4f}'
        if i == best_acc_pos:
            label += ' ★'
        axes[0].text(i, v + s + 0.008, label, ha='center', fontsize=9, fontweight='bold')

    # F1-Score Macro
    axes[1].bar(x, recap_df['mean_f1_macro'], yerr=recap_df['std_f1_macro'],
                capsize=5, color=colors_f1, edgecolor='black', linewidth=0.5)
    axes[1].set_ylabel('F1-Score Macro', fontsize=11)
    axes[1].set_title('F1-Score Macro (Mean ± Std)', fontsize=12, fontweight='bold')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(configs, rotation=45, ha='right', fontsize=10)
    y_min_f1 = max(0, recap_df['mean_f1_macro'].min() - 0.15)
    axes[1].set_ylim(y_min_f1, 1.05)
    for i, (v, s) in enumerate(zip(recap_df['mean_f1_macro'], recap_df['std_f1_macro'])):
        label = f'{v:.4f}'
        if i == best_f1_pos:
            label += ' ★'
        axes[1].text(i, v + s + 0.008, label, ha='center', fontsize=9, fontweight='bold')

    plt.suptitle(stage_name, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir, 'grafik_perbandingan.png'), dpi=150, bbox_inches='tight')
    plt.close(fig)


print('✅ Semua fungsi berhasil didefinisikan.')

✅ Semua fungsi berhasil didefinisikan.


## 4. Stage Runner — Menjalankan 1 Tahap Tuning

In [7]:
def run_stage(X, y, class_names, stage_name, stage_dir,
              param_variations, base_params, value_map):
    """
    Menjalankan satu tahap tuning lengkap (semua variasi x K-fold).
    Support resume: jika rekapitulasi.csv sudah ada, tahap ini di-skip.
    """

    os.makedirs(stage_dir, exist_ok=True)
    recap_path = os.path.join(stage_dir, 'rekapitulasi.csv')

    # Resume: jika sudah selesai, skip
    if os.path.exists(recap_path):
        print(f'\n{"="*60}')
        print(f'⏭ {stage_name} — SUDAH SELESAI SEBELUMNYA (skip)')
        print(f'{"="*60}')
        recap = pd.read_csv(recap_path)
        best_config = recap.loc[recap['mean_f1_macro'].idxmax(), 'config']
        best_val = value_map[best_config]
        print(f'  🏆 Terbaik: {best_config} → F1={recap.loc[recap["mean_f1_macro"].idxmax(), "mean_f1_macro"]:.4f}')
        return recap, best_val

    print(f'\n{"="*60}')
    print(f'🔬 {stage_name}')
    print(f'{"="*60}')

    all_fold_results = []
    stage_start = time.time()

    for var_name, var_overrides in param_variations.items():
        params = {**base_params, **var_overrides}

        print(f'\n  📋 Konfigurasi: {var_name}')
        print(f'     lr={params["learning_rate"]}, bs={params["batch_size"]}, '
              f'dropout={params["dropout_rate"]}, epochs={params["epochs"]}, '
              f'aug={params.get("augmentation", "baseline")}')

        fold_results = run_kfold(X, y, class_names, params, var_name, stage_dir)
        all_fold_results.extend(fold_results)

    # Simpan hasil per fold
    df_folds = pd.DataFrame(all_fold_results)
    df_folds.to_csv(os.path.join(stage_dir, 'hasil_per_fold.csv'), index=False)

    # Rekapitulasi
    recap = df_folds.groupby('config').agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_f1_macro=('f1_macro', 'mean'),
        std_f1_macro=('f1_macro', 'std'),
        mean_precision=('precision_macro', 'mean'),
        std_precision=('precision_macro', 'std'),
        mean_recall=('recall_macro', 'mean'),
        std_recall=('recall_macro', 'std'),
        mean_epochs=('epochs_run', 'mean'),
        mean_val_loss=('best_val_loss', 'mean'),
        total_time_seconds=('time_seconds', 'sum'),
    ).reset_index()

    for col in recap.columns:
        if recap[col].dtype == 'float64':
            recap[col] = recap[col].round(4)

    recap = recap.sort_values('mean_f1_macro', ascending=False)
    recap.to_csv(recap_path, index=False)

    # Terbaik
    best_row = recap.iloc[0]
    best_config = best_row['config']
    best_val = value_map[best_config]
    stage_elapsed = time.time() - stage_start

    print(f'\n  {"─"*50}')
    print(f'  🏆 TERBAIK: {best_config}')
    print(f'     Accuracy : {best_row["mean_accuracy"]:.4f} ± {best_row["std_accuracy"]:.4f}')
    print(f'     F1-Score : {best_row["mean_f1_macro"]:.4f} ± {best_row["std_f1_macro"]:.4f}')
    print(f'     Precision: {best_row["mean_precision"]:.4f} ± {best_row["std_precision"]:.4f}')
    print(f'     Recall   : {best_row["mean_recall"]:.4f} ± {best_row["std_recall"]:.4f}')
    print(f'     Waktu    : {stage_elapsed/60:.1f} menit')

    # Tabel
    print(f'\n  📊 Rekapitulasi {stage_name}:')
    display_cols = ['config', 'mean_accuracy', 'std_accuracy',
                    'mean_f1_macro', 'std_f1_macro', 'mean_epochs']
    print(recap[display_cols].to_string(index=False))

    # Grafik
    if len(recap) > 1:
        _save_comparison_chart(recap, stage_name, stage_dir)

    return recap, best_val


def save_final_summary(output_dir, current_best, total_time, stage_recaps):
    """Buat ringkasan akhir: JSON, TXT, dan CSV gabungan."""

    # JSON
    with open(os.path.join(output_dir, 'konfigurasi_terbaik.json'), 'w') as f:
        json.dump(current_best, f, indent=2, ensure_ascii=False)

    # TXT
    with open(os.path.join(output_dir, 'ringkasan_tuning.txt'), 'w', encoding='utf-8') as f:
        f.write('=' * 60 + '\n')
        f.write('RINGKASAN HYPERPARAMETER TUNING MOBILENETV2\n')
        f.write('Stratified K-Fold Cross-Validation (Control Variable Method)\n')
        f.write('=' * 60 + '\n\n')
        f.write(f'Dataset      : {DATASET_DIR}\n')
        f.write(f'K-Fold       : {K_FOLDS}\n')
        f.write(f'Seed         : {SEED}\n')
        f.write(f'Image Size   : {IMG_SIZE}\n')
        f.write(f'Total Waktu  : {total_time/60:.1f} menit ({total_time/3600:.1f} jam)\n\n')
        f.write('KONFIGURASI TERBAIK (FINAL):\n')
        f.write('-' * 40 + '\n')
        for k, v in current_best.items():
            f.write(f'  {k}: {v}\n')
        f.write('\nHASIL PER TAHAP:\n')
        f.write('-' * 40 + '\n')
        for stage_name, recap_df in stage_recaps:
            best = recap_df.iloc[0]
            f.write(f'\n  {stage_name}:\n')
            f.write(f'    Terbaik   : {best["config"]}\n')
            f.write(f'    Accuracy  : {best["mean_accuracy"]:.4f} ± {best["std_accuracy"]:.4f}\n')
            f.write(f'    F1 Macro  : {best["mean_f1_macro"]:.4f} ± {best["std_f1_macro"]:.4f}\n')

    # CSV gabungan
    all_recaps = []
    for stage_name, recap_df in stage_recaps:
        recap_copy = recap_df.copy()
        recap_copy.insert(0, 'tahap', stage_name)
        recap_copy['terbaik'] = ''
        recap_copy.iloc[0, recap_copy.columns.get_loc('terbaik')] = '★'
        all_recaps.append(recap_copy)

    combined = pd.concat(all_recaps, ignore_index=True)
    combined.to_csv(os.path.join(output_dir, 'rekapitulasi_semua_tahap.csv'), index=False)

    print(f'\n📁 File tersimpan di: {output_dir}')
    print(f'   📄 konfigurasi_terbaik.json')
    print(f'   📄 ringkasan_tuning.txt')
    print(f'   📄 rekapitulasi_semua_tahap.csv')


print('✅ Stage runner & summary functions ready.')

✅ Stage runner & summary functions ready.


## 5. 🚀 Jalankan Pipeline Tuning

Cell ini menjalankan **seluruh 6 tahap** secara berurutan.  
Jika terputus, jalankan ulang — tahap yang sudah selesai akan otomatis di-skip.

In [8]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(os.path.join(OUTPUT_DIR, 'class_names.json'), 'w') as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)

total_start = time.time()

# Parameter awal
current_best = {
    'learning_rate': 1e-3,
    'batch_size': 32,
    'dropout_rate': 0.2,
    'epochs': 30,
    'augmentation': 'baseline'
}

stage_recaps = []

# ══════════════════════════════════════════════════════════
# TAHAP 0: BASELINE
# ══════════════════════════════════════════════════════════
recap, _ = run_stage(
    X, y, class_names,
    stage_name='Tahap 0: Baseline',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_0_baseline'),
    param_variations={'baseline': {}},
    base_params=current_best,
    value_map={'baseline': None}
)
stage_recaps.append(('Tahap 0: Baseline', recap))

# ══════════════════════════════════════════════════════════
# TAHAP 1: LEARNING RATE
# ══════════════════════════════════════════════════════════
recap, best_lr = run_stage(
    X, y, class_names,
    stage_name='Tahap 1: Learning Rate',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_1_learning_rate'),
    param_variations={
        'lr=1e-3': {'learning_rate': 1e-3},
        'lr=1e-4': {'learning_rate': 1e-4},
        'lr=1e-5': {'learning_rate': 1e-5},
    },
    base_params=current_best,
    value_map={'lr=1e-3': 1e-3, 'lr=1e-4': 1e-4, 'lr=1e-5': 1e-5}
)
current_best['learning_rate'] = best_lr
stage_recaps.append(('Tahap 1: Learning Rate', recap))
print(f'\n  🔒 Learning Rate di-lock: {best_lr}')

# ══════════════════════════════════════════════════════════
# TAHAP 2: BATCH SIZE
# ══════════════════════════════════════════════════════════
recap, best_bs = run_stage(
    X, y, class_names,
    stage_name='Tahap 2: Batch Size',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_2_batch_size'),
    param_variations={
        'bs=16': {'batch_size': 16},
        'bs=32': {'batch_size': 32},
        'bs=64': {'batch_size': 64},
    },
    base_params=current_best,
    value_map={'bs=16': 16, 'bs=32': 32, 'bs=64': 64}
)
current_best['batch_size'] = best_bs
stage_recaps.append(('Tahap 2: Batch Size', recap))
print(f'\n  🔒 Batch Size di-lock: {best_bs}')

# ══════════════════════════════════════════════════════════
# TAHAP 3: DROPOUT RATE
# ══════════════════════════════════════════════════════════
recap, best_dropout = run_stage(
    X, y, class_names,
    stage_name='Tahap 3: Dropout Rate',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_3_dropout'),
    param_variations={
        'dropout=0.2': {'dropout_rate': 0.2},
        'dropout=0.3': {'dropout_rate': 0.3},
        'dropout=0.5': {'dropout_rate': 0.5},
    },
    base_params=current_best,
    value_map={'dropout=0.2': 0.2, 'dropout=0.3': 0.3, 'dropout=0.5': 0.5}
)
current_best['dropout_rate'] = best_dropout
stage_recaps.append(('Tahap 3: Dropout Rate', recap))
print(f'\n  🔒 Dropout Rate di-lock: {best_dropout}')

# ══════════════════════════════════════════════════════════
# TAHAP 4: MAX EPOCHS
# ══════════════════════════════════════════════════════════
recap, best_epochs = run_stage(
    X, y, class_names,
    stage_name='Tahap 4: Max Epochs',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_4_epochs'),
    param_variations={
        'epochs=15': {'epochs': 15},
        'epochs=30': {'epochs': 30},
        'epochs=50': {'epochs': 50},
    },
    base_params=current_best,
    value_map={'epochs=15': 15, 'epochs=30': 30, 'epochs=50': 50}
)
current_best['epochs'] = best_epochs
stage_recaps.append(('Tahap 4: Max Epochs', recap))
print(f'\n  🔒 Max Epochs di-lock: {best_epochs}')

# ══════════════════════════════════════════════════════════
# TAHAP 5: DATA AUGMENTATION
# ══════════════════════════════════════════════════════════
recap, best_aug = run_stage(
    X, y, class_names,
    stage_name='Tahap 5: Data Augmentation',
    stage_dir=os.path.join(OUTPUT_DIR, 'tahap_5_augmentation'),
    param_variations={
        'aug=minimal': {'augmentation': 'minimal'},
        'aug=baseline': {'augmentation': 'baseline'},
        'aug=enhanced': {'augmentation': 'enhanced'},
    },
    base_params=current_best,
    value_map={'aug=minimal': 'minimal', 'aug=baseline': 'baseline', 'aug=enhanced': 'enhanced'}
)
current_best['augmentation'] = best_aug
stage_recaps.append(('Tahap 5: Data Augmentation', recap))
print(f'\n  🔒 Augmentation di-lock: {best_aug}')

# ══════════════════════════════════════════════════════════
# RINGKASAN FINAL
# ══════════════════════════════════════════════════════════
total_time = time.time() - total_start

print(f'\n{"="*60}')
print(f'✅ SELURUH TUNING SELESAI!')
print(f'   Total waktu: {total_time/60:.1f} menit ({total_time/3600:.1f} jam)')
print(f'{"="*60}')
print(f'\n🏆 KONFIGURASI TERBAIK (FINAL):')
print(f'   Learning Rate  : {current_best["learning_rate"]}')
print(f'   Batch Size     : {current_best["batch_size"]}')
print(f'   Dropout Rate   : {current_best["dropout_rate"]}')
print(f'   Max Epochs     : {current_best["epochs"]}')
print(f'   Augmentation   : {current_best["augmentation"]}')

save_final_summary(OUTPUT_DIR, current_best, total_time, stage_recaps)

print(f'\n🎓 Semua hasil siap untuk Bab 4 skripsi!')


🔬 Tahap 0: Baseline

  📋 Konfigurasi: baseline
     lr=0.001, bs=32, dropout=0.2, epochs=30, aug=baseline
    Fold 1/5... 

I0000 00:00:1784910011.814718      59 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1784910011.817540      59 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Acc=0.9125  F1=0.9115  Epochs=28  (55.5s)
    Fold 2/5... Acc=0.9625  F1=0.9623  Epochs=30  (53.8s)
    Fold 3/5... Acc=0.9313  F1=0.9302  Epochs=30  (55.2s)
    Fold 4/5... Acc=0.9500  F1=0.9500  Epochs=25  (47.3s)
    Fold 5/5... Acc=0.9375  F1=0.9364  Epochs=27  (50.3s)

  ──────────────────────────────────────────────────
  🏆 TERBAIK: baseline
     Accuracy : 0.9388 ± 0.0190
     F1-Score : 0.9381 ± 0.0194
     Precision: 0.9408 ± 0.0178
     Recall   : 0.9388 ± 0.0190
     Waktu    : 4.4 menit

  📊 Rekapitulasi Tahap 0: Baseline:
  config  mean_accuracy  std_accuracy  mean_f1_macro  std_f1_macro  mean_epochs
baseline         0.9388         0.019         0.9381        0.0194         28.0

🔬 Tahap 1: Learning Rate

  📋 Konfigurasi: lr=1e-3
     lr=0.001, bs=32, dropout=0.2, epochs=30, aug=baseline
    Fold 1/5... Acc=0.9187  F1=0.9185  Epochs=30  (55.2s)
    Fold 2/5... Acc=0.9500  F1=0.9503  Epochs=20  (40.5s)
    Fold 3/5... Acc=0.9

2026-07-24 17:14:40.792477: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-24 17:14:40.935685: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1784913283.389432     135 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-07-24 17:15:16.590190: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-24 17:15:16.727589: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-o

Acc=0.9313  F1=0.9307  Epochs=18  (55.6s)
    Fold 2/5... Acc=0.9375  F1=0.9377  Epochs=20  (42.3s)
    Fold 3/5... Acc=0.9437  F1=0.9439  Epochs=23  (45.7s)
    Fold 4/5... Acc=0.9375  F1=0.9373  Epochs=28  (50.8s)
    Fold 5/5... Acc=0.9563  F1=0.9565  Epochs=22  (44.3s)

  📋 Konfigurasi: aug=baseline
     lr=0.001, bs=16, dropout=0.3, epochs=50, aug=baseline
    Fold 1/5... Acc=0.9187  F1=0.9189  Epochs=19  (43.1s)
    Fold 2/5... Acc=0.9812  F1=0.9812  Epochs=28  (56.4s)
    Fold 3/5... Acc=0.9437  F1=0.9435  Epochs=28  (57.3s)
    Fold 4/5... Acc=0.9500  F1=0.9500  Epochs=18  (40.2s)
    Fold 5/5... Acc=0.9313  F1=0.9312  Epochs=16  (38.1s)

  📋 Konfigurasi: aug=enhanced
     lr=0.001, bs=16, dropout=0.3, epochs=50, aug=enhanced
    Fold 1/5... Acc=0.9125  F1=0.9117  Epochs=27  (58.5s)
    Fold 2/5... Acc=0.9688  F1=0.9686  Epochs=20  (45.4s)
    Fold 3/5... Acc=0.9563  F1=0.9565  Epochs=21  (47.3s)
    Fold 4/5... Acc=0.9688  F1=0.9686  Epochs=31  (63.2s)
    Fold 5/5... Acc=0.93

## 6. Lihat Hasil

Jalankan cell di bawah untuk melihat rekapitulasi semua tahap dan konfigurasi terbaik.

In [9]:
# Tampilkan rekapitulasi semua tahap
csv_path = os.path.join(OUTPUT_DIR, 'rekapitulasi_semua_tahap.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('📊 REKAPITULASI SEMUA TAHAP')
    print('=' * 80)
    display(df)
else:
    print('⚠ Belum ada hasil. Jalankan cell pipeline di atas terlebih dahulu.')

# Tampilkan konfigurasi terbaik
json_path = os.path.join(OUTPUT_DIR, 'konfigurasi_terbaik.json')
if os.path.exists(json_path):
    with open(json_path, 'r') as f:
        best = json.load(f)
    print('\n🏆 KONFIGURASI TERBAIK:')
    for k, v in best.items():
        print(f'   {k}: {v}')

📊 REKAPITULASI SEMUA TAHAP


,tahap,config,mean_accuracy,std_accuracy,mean_f1_macro,std_f1_macro,mean_precision,std_precision,mean_recall,std_recall,mean_epochs,mean_val_loss,total_time_seconds,terbaik
0,Tahap 0: Baseline,baseline,0.9388,0.0190,0.9381,0.0194,0.9408,0.0178,0.9388,0.0190,28.0,0.1513,262.1,★
1,Tahap 1: Learning Rate,lr=1e-3,0.9350,0.0151,0.9347,0.0154,0.9386,0.0125,0.9350,0.0151,25.2,0.1658,240.3,★
2,Tahap 1: Learning Rate,lr=1e-4,0.8950,0.0291,0.8932,0.0311,0.8999,0.0241,0.8950,0.0291,30.0,0.3674,283.1,NaN
3,Tahap 1: Learning Rate,lr=1e-5,0.3538,0.0754,0.3452,0.0730,0.3887,0.0824,0.3538,0.0754,30.0,1.7214,282.1,NaN
4,Tahap 2: Batch Size,bs=16,0.9463,0.0210,0.9460,0.0212,0.9477,0.0200,0.9463,0.0210,21.4,0.1500,227.0,★
5,Tahap 2: Batch Size,bs=32,0.9438,0.0238,0.9434,0.0242,0.9472,0.0203,0.9437,0.0238,24.8,0.1668,240.3,NaN
6,Tahap 2: Batch Size,bs=64,0.9387,0.0120,0.9381,0.0127,0.9418,0.0084,0.9387,0.0120,29.8,0.1722,278.8,NaN
7,Tahap 3: Dropout Rate,dropout=0.3,0.9513,0.0162,0.9510,0.0163,0.9530,0.0163,0.9513,0.0162,23.8,0.1366,248.8,★
8,Tahap 3: Dropout Rate,dropout=0.5,0.9463,0.0095,0.9462,0.0094,0.9478,0.0089,0.9462,0.0095,20.6,0.1514,224.2,NaN
9,Tahap 3: Dropout Rate,dropout=0.2,0.9425,0.0112,0.9422,0.0114,0.9450,0.0080,0.9425,0.0112,22.2,0.1417,234.7,NaN



🏆 KONFIGURASI TERBAIK:
   learning_rate: 0.001
   batch_size: 16
   dropout_rate: 0.3
   epochs: 50
   augmentation: enhanced


## 7. Download Hasil

Jalankan cell di bawah untuk **zip semua hasil** agar mudah di-download dari Kaggle.

In [10]:
import shutil

zip_path = '/kaggle/working/hasil_tuning_vision'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print(f'✅ File zip tersedia di: {zip_path}.zip')
print('   Klik Output tab di sidebar kanan Kaggle untuk download.')

✅ File zip tersedia di: /kaggle/working/hasil_tuning_vision.zip
   Klik Output tab di sidebar kanan Kaggle untuk download.
